### Importando as bibliotecas e criação da função que converte os números do padrão brasileiro para o padrão do bano de dados

In [2]:
import pdfplumber
import pandas as pd
import os

def limpar_numero(valor):
    
    if not valor:
        return 0.0
    valor_str = str(valor).replace('.', '').replace(',', '.').strip()
    try:
        return float(valor_str)
    except ValueError:
        return 0.0

# Teste rápido para ver se a função está operando corretamente
teste_numero = limpar_numero("5,38")
print(f"Resultado do teste: {teste_numero} | Tipo: {type(teste_numero)}")

Resultado do teste: 5.38 | Tipo: <class 'float'>


# Extração do PDF

### Teste com o primeiro PDF:

In [ ]:
# Ajuste o nome do arquivo para corresponder exatamente ao que está na sua pasta
caminho_teste = "CandidatoVaga/candidato-vaga-2018.pdf" 

with pdfplumber.open(caminho_teste) as pdf:
    # Pegamos apenas a primeira página
    pagina_teste = pdf.pages[0] 
    tabela_bruta = pagina_teste.extract_table()
    
    print(f"Total de linhas extraídas na página 1: {len(tabela_bruta)}\n")
   
    # Imprime a linha 4 para validarmos se há 16 posições (índices de 0 a 15)
    print(tabela_bruta[0])
    print(tabela_bruta[1])
   

Total de linhas extraídas na página 1: 23

['Administração (RIO)', '66', '355', '5,38', '24', '24', '1,00', '24', '15', '0,63', '6', '0', '0,00', '120', '394', '3,28']
['Arqueologia (RIO)', '16', '58', '3,63', '6', '1', '0,17', '6', '3', '0,50', '2', '0', '0,00', '30', '62', '2,07']
['Arquitetura e Urbanismo\n(PETRÓPOLIS)', '26', '348', '13,38', '10', '9', '0,90', '10', '6', '0,60', '4', '0', '0,00', '50', '363', '7,26']


### Esta forma nos permite entender como ficará a estrutura de cada curso, até cada posição receber um nome.

C:\Users\José\OneDrive\Documentos\projeto-vestibular\projeto_vestibular\Captura de tela 2026-09-14 213516.png

### Crição do dicionário

In [25]:
def extrair_vagas_dicionario(caminho_pdf, ano):
    dados_finais = []
    
    with pdfplumber.open(caminho_pdf) as pdf:
        for pagina in pdf.pages:
            tabela = pagina.extract_table()
            if not tabela:
                continue
                
            for linha in tabela[0:]: 
                if not linha or not linha[0] or "TOTAL" in str(linha[0]):
                    continue
                    
                curso_nome = str(linha[0]).replace('\n', ' ').strip()
                
                try:
                    # O sdicionário estruturado
                    dados_curso = {
                        "Não Reservada": {"vagas": linha[1], "inscritos": linha[2], "relacao": linha[3]},
                        "Rede Pública": {"vagas": linha[4], "inscritos": linha[5], "relacao": linha[6]},
                        "Negros/Indígenas": {"vagas": linha[7], "inscritos": linha[8], "relacao": linha[9]},
                        "Pessoas com Deficiência": {"vagas": linha[10], "inscritos": linha[11], "relacao": linha[12]},
                        "Geral": {"vagas": linha[13], "inscritos": linha[14], "relacao": linha[15]}
                    }
                    
                    # Achatando para a Tabela Longa
                    for categoria, metricas in dados_curso.items():
                        dados_finais.append({
                            'ano_vestibular': int(ano),
                            'curso': curso_nome,
                            'categoria_cota': categoria,
                            'vagas': limpar_numero(metricas["vagas"]),
                            'inscritos': limpar_numero(metricas["inscritos"]),
                            'relacao_cand_vaga': limpar_numero(metricas["relacao"])
                        })
                except IndexError:
                    pass
                    
    return pd.DataFrame(dados_finais)

# Executando apenas para o arquivo de teste
df_2018 = extrair_vagas_dicionario(caminho_teste, "2018")
display(df_2018.head(10))

,ano_vestibular,curso,categoria_cota,vagas,inscritos,relacao_cand_vaga
0,2018,Administração (RIO),Não Reservada,66.0,355.0,5.38
1,2018,Administração (RIO),Rede Pública,24.0,24.0,1.00
2,2018,Administração (RIO),Negros/Indígenas,24.0,15.0,0.63
3,2018,Administração (RIO),Pessoas com Deficiência,6.0,0.0,0.00
4,2018,Administração (RIO),Geral,120.0,394.0,3.28
5,2018,Arqueologia (RIO),Não Reservada,16.0,58.0,3.63
6,2018,Arqueologia (RIO),Rede Pública,6.0,1.0,0.17
7,2018,Arqueologia (RIO),Negros/Indígenas,6.0,3.0,0.50
8,2018,Arqueologia (RIO),Pessoas com Deficiência,2.0,0.0,0.00
9,2018,Arqueologia (RIO),Geral,30.0,62.0,2.07


In [26]:
pasta_vagas = "CandidatoVaga"
lista_dfs = []

# Lista todos os PDFs da pasta
arquivos_pdf = [f for f in os.listdir(pasta_vagas) if f.endswith('.pdf')]
print(f"Iniciando processamento de {len(arquivos_pdf)} arquivos...\n")

for arquivo in arquivos_pdf:
    print(f"Extraindo tabela de: {arquivo}...")
    caminho_completo = os.path.join(pasta_vagas, arquivo)
    
    # Extrai o ano do nome do arquivo (ajuste o corte se o seu padrão de nome for diferente)
    # Exemplo: se o nome for "candidato-vaga-2018.pdf", pega os 4 últimos caracteres antes do ".pdf"
    ano = arquivo.replace('.pdf', '')[-4:] 
    
    df_ano = extrair_vagas_dicionario(caminho_completo, ano)
    lista_dfs.append(df_ano)

# Consolida o banco final
df_candidato_vaga = pd.concat(lista_dfs, ignore_index=True)
print("\nProcessamento concluído!")
df_candidato_vaga.info()

Iniciando processamento de 10 arquivos...

Extraindo tabela de: candidato-vaga-2018.pdf...
Extraindo tabela de: candidato-vaga-2019.pdf...
Extraindo tabela de: candidato-vaga-2020.pdf...
Extraindo tabela de: candidato-vaga-2021.pdf...
Extraindo tabela de: candidato-vaga-2022.pdf...
Extraindo tabela de: candidato-vaga-2023.pdf...
Extraindo tabela de: candidato-vaga-2024.pdf...
Extraindo tabela de: candidato-vaga-2025.pdf...
Extraindo tabela de: candidato-vaga-2026.pdf...
Extraindo tabela de: candidato-vaga-uezo-2018.pdf...

Processamento concluído!
<class 'pandas.DataFrame'>
RangeIndex: 3185 entries, 0 to 3184
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ano_vestibular     3185 non-null   int64  
 1   curso              3185 non-null   str    
 2   categoria_cota     3185 non-null   str    
 3   vagas              3185 non-null   float64
 4   inscritos          3185 non-null   float64
 5   relacao_ca